## Loading Files

In [1]:
from safetensors import safe_open
import os

files = [
    "./my_models/model-00001-of-00002.safetensors",
    "./my_models/model-00002-of-00002.safetensors",
]

for file in files:
    print(f"\n=== {file} ===")

    with safe_open(f"{file}", framework="pt", device="cpu") as f:
        for key in f.keys():
            print(f"Name: {key}, shape:{f.get_tensor(key).shape}")


=== ./my_models/model-00001-of-00002.safetensors ===
Name: model.embed_tokens.weight, shape:torch.Size([128256, 3072])
Name: model.layers.0.input_layernorm.weight, shape:torch.Size([3072])
Name: model.layers.0.mlp.down_proj.weight, shape:torch.Size([3072, 8192])
Name: model.layers.0.mlp.gate_proj.weight, shape:torch.Size([8192, 3072])
Name: model.layers.0.mlp.up_proj.weight, shape:torch.Size([8192, 3072])
Name: model.layers.0.post_attention_layernorm.weight, shape:torch.Size([3072])
Name: model.layers.0.self_attn.k_proj.weight, shape:torch.Size([1024, 3072])
Name: model.layers.0.self_attn.o_proj.weight, shape:torch.Size([3072, 3072])
Name: model.layers.0.self_attn.q_proj.weight, shape:torch.Size([3072, 3072])
Name: model.layers.0.self_attn.v_proj.weight, shape:torch.Size([1024, 3072])
Name: model.layers.1.input_layernorm.weight, shape:torch.Size([3072])
Name: model.layers.1.mlp.down_proj.weight, shape:torch.Size([3072, 8192])
Name: model.layers.1.mlp.gate_proj.weight, shape:torch.Size

## Rebuilding Modules

In [2]:
import torch

torch.set_num_threads(4)
torch.set_num_interop_threads(1)


class LlamaMlp(torch.nn.Module):
    def __init__(self, hidden_size, intermediate_size):
        super().__init__()
        self.up_proj = torch.nn.Linear(
            in_features=hidden_size, out_features=intermediate_size, bias=False
        )
        self.gate_proj = torch.nn.Linear(
            in_features=hidden_size, out_features=intermediate_size, bias=False
        )
        self.down_proj = torch.nn.Linear(
            in_features=intermediate_size, out_features=hidden_size, bias=False
        )

    def forward(self, x):
        up = self.up_proj(x)
        gate = self.gate_proj(x)

        hidden = torch.nn.functional.silu(gate) * up

        return self.down_proj(hidden)

In [118]:
class LlamaAttention(torch.nn.Module):
    def __init__(self, num_kv_head, num_attn_head, head_dim, hidden_size, max_seq_len):
        super().__init__()

        self.num_kv_head = num_kv_head
        self.num_attn_head = num_attn_head
        self.head_dim = head_dim
        self.hidden_size = hidden_size
        self.max_seq_len = max_seq_len

        self.theta = 500000
        self.low_freq_factor = 1.0
        self.high_freq_factor = 4.0
        self.factor = 32.0
        self.original_max_position_embeddings = 8192

        self.k_proj = torch.nn.Linear(
            in_features=hidden_size, out_features=num_kv_head * head_dim, bias=False
        )
        self.o_proj = torch.nn.Linear(
            in_features=hidden_size, out_features=hidden_size, bias=False
        )
        self.q_proj = torch.nn.Linear(
            in_features=hidden_size, out_features=hidden_size, bias=False
        )
        self.v_proj = torch.nn.Linear(
            in_features=hidden_size, out_features=num_kv_head * head_dim, bias=False
        )

        self.register_buffer(
            "mask", torch.triu(torch.ones(max_seq_len, max_seq_len), diagonal=1).bool()
        )

    def forward(self, x, past_kv=None):

        EB, ES, ED = x.shape

        q = self.q_proj(x)
        k = self.k_proj(x)
        v = self.v_proj(x)

        q = q.view(EB, ES, self.num_attn_head, self.head_dim).transpose(1, 2)
        k = k.view(EB, ES, self.num_kv_head, self.head_dim).transpose(1, 2)
        v = v.view(EB, ES, self.num_kv_head, self.head_dim).transpose(1, 2)

        if past_kv is not None:
            past_k, past_v = past_kv

            k = torch.cat([past_k, k], dim=2)
            v = torch.cat([past_v, v], dim=2)

        present_kv = (k, v)
        KB, KH, KS, KD = k.shape

        # Llama RoPE
        inv_freq = 1.0 / (
            self.theta
            ** (
                torch.arange(
                    0,
                    self.head_dim,
                    2,
                    device=x.device,
                    dtype=torch.float32,
                )
                / self.head_dim
            )
        )

        wavelength = 2 * torch.pi / inv_freq

        low_freq_wavelen = self.original_max_position_embeddings / self.low_freq_factor

        high_freq_wavelen = (
            self.original_max_position_embeddings / self.high_freq_factor
        )

        inv_freq = torch.where(
            wavelength > low_freq_wavelen,
            inv_freq / self.factor,
            inv_freq,
        )

        smooth_factor = (
            self.original_max_position_embeddings / wavelength - self.low_freq_factor
        ) / (self.high_freq_factor - self.low_freq_factor)

        smoothed_inv_freq = (1 - smooth_factor) * (
            inv_freq / self.factor
        ) + smooth_factor * inv_freq

        is_medium = (wavelength >= high_freq_wavelen) & (wavelength <= low_freq_wavelen)

        inv_freq = torch.where(
            is_medium,
            smoothed_inv_freq,
            inv_freq,
        )

        past_len = 0 if past_kv is None else past_kv[0].shape[2]

        q_positions = torch.arange(
            past_len,
            past_len + ES,
            device=x.device,
            dtype=torch.float32,
        )

        k_positions = torch.arange(
            KS,
            device=x.device,
            dtype=torch.float32,
        )

        q_freqs = torch.outer(q_positions, inv_freq)
        k_freqs = torch.outer(k_positions, inv_freq)

        q_cos = torch.cat((q_freqs.cos(), q_freqs.cos()), dim=-1)
        q_sin = torch.cat((q_freqs.sin(), q_freqs.sin()), dim=-1)

        k_cos = torch.cat((k_freqs.cos(), k_freqs.cos()), dim=-1)
        k_sin = torch.cat((k_freqs.sin(), k_freqs.sin()), dim=-1)

        q = (q * q_cos) + (self.rotate_half(q) * q_sin)
        k = (k * k_cos) + (self.rotate_half(k) * k_sin)

        num_groups = self.num_attn_head // self.num_kv_head

        k = k.repeat_interleave(num_groups, dim=1)
        v = v.repeat_interleave(num_groups, dim=1)

        scores = torch.einsum("bhqd, bhkd -> bhqk", q, k) * (self.head_dim**-0.5)
        if ES > 1:
            scores = scores.masked_fill(self.mask[:ES, :KS], float("-inf"))

        attn_weights = torch.softmax(scores, dim=-1)

        attn_output = torch.einsum("bhqk, bhkd -> bhqd", attn_weights, v)
        attn_output = attn_output.transpose(1, 2).contiguous()
        attn_output = attn_output.view(EB, ES, self.hidden_size)

        output = self.o_proj(attn_output)

        return output, present_kv

    def rotate_half(self, x):
        x1 = x[..., : self.head_dim // 2]
        x2 = x[..., self.head_dim // 2 :]

        return torch.cat((-x2, x1), dim=-1)

In [125]:
class LlamaDecoderLayer(torch.nn.Module):
    def __init__(
        self,
        hidden_size,
        intermediate_size,
        num_kv_head,
        num_attn_head,
        head_dim,
        max_seq_len,
    ):
        super().__init__()
        self.input_layernorm = torch.nn.RMSNorm(hidden_size, eps=1e-5)
        self.self_attn = LlamaAttention(
            num_kv_head,
            num_attn_head,
            head_dim,
            hidden_size,
            max_seq_len,
        )
        self.post_attention_layernorm = torch.nn.RMSNorm(hidden_size, eps=1e-5)
        self.mlp = LlamaMlp(
            hidden_size,
            intermediate_size,
        )

    def forward(self, x, past_kv=None):

        residual = x

        x = self.input_layernorm(x)
        x, present_kv = self.self_attn(x, past_kv)

        x = residual + x

        residual = x

        x = self.post_attention_layernorm(x)
        x = self.mlp(x)

        x = residual + x

        return x, present_kv

In [155]:
class LlamaModel(torch.nn.Module):
    def __init__(
        self,
        files,
        num_layers,
        hidden_size,
        intermediate_size,
        num_kv_head,
        num_attn_head,
        head_dim,
        max_seq_len,
    ):
        super().__init__()

        self.num_layers = num_layers

        self.handles = [safe_open(path, framework="pt", device="cpu") for path in files]

        self.embed_tokens = torch.nn.Embedding(
            num_embeddings=128256, embedding_dim=3072
        )
        self.embed_tokens.weight.data.copy_(
            self.get_tensor(self.handles, "model.embed_tokens.weight")
        )

        self.decoder = LlamaDecoderLayer(
            hidden_size,
            intermediate_size,
            num_kv_head,
            num_attn_head,
            head_dim,
            max_seq_len,
        ).eval()

        self.norm = torch.nn.RMSNorm(hidden_size, eps=1e-5)
        self.norm.weight.data.copy_(self.get_tensor(self.handles, "model.norm.weight"))

        # KV cache
        self.kv_cache = [None] * num_layers

    @torch.no_grad()
    def forward(self, input_ids):

        x = self.embed_tokens(input_ids)

        for layer_id in range(self.num_layers):

            self.load_layer(self.decoder, layer_id, self.handles)
            x, self.kv_cache[layer_id] = self.decoder(x, self.kv_cache[layer_id])

        x = self.norm(x)

        logits = x @ self.embed_tokens.weight.T

        next_token_logits = logits[:, -1, :]

        return torch.argmax(next_token_logits, dim=-1)

    def get_tensor(self, handles, name):
        for handle in handles:
            if name in handle.keys():
                return handle.get_tensor(name)

        raise KeyError(f"Tensor not found: {name}")

    def load_layer(self, decoder, layer_id, handles):

        prefix = f"model.layers.{layer_id}."

        decoder.input_layernorm.weight.data.copy_(
            self.get_tensor(handles, prefix + "input_layernorm.weight")
        )

        decoder.self_attn.q_proj.weight.data.copy_(
            self.get_tensor(handles, prefix + "self_attn.q_proj.weight")
        )

        decoder.self_attn.k_proj.weight.data.copy_(
            self.get_tensor(handles, prefix + "self_attn.k_proj.weight")
        )

        decoder.self_attn.v_proj.weight.data.copy_(
            self.get_tensor(handles, prefix + "self_attn.v_proj.weight")
        )

        decoder.self_attn.o_proj.weight.data.copy_(
            self.get_tensor(handles, prefix + "self_attn.o_proj.weight")
        )

        decoder.post_attention_layernorm.weight.data.copy_(
            self.get_tensor(handles, prefix + "post_attention_layernorm.weight")
        )

        decoder.mlp.gate_proj.weight.data.copy_(
            self.get_tensor(handles, prefix + "mlp.gate_proj.weight")
        )

        decoder.mlp.up_proj.weight.data.copy_(
            self.get_tensor(handles, prefix + "mlp.up_proj.weight")
        )

        decoder.mlp.down_proj.weight.data.copy_(
            self.get_tensor(handles, prefix + "mlp.down_proj.weight")
        )

    def clear_kv_cache(self):
        self.kv_cache = [None] * self.num_layers

In [156]:
# let's load LlamaDecoderLayer for 32 times

hidden_size = 3072
head_dim = 128
num_attn_head = 24
num_kv_head = 8
max_seq_len = 100
theta = 500000
intermediate_size = 8192
layers = 28

model = LlamaModel(
    files,
    layers,
    hidden_size,
    intermediate_size,
    num_kv_head,
    num_attn_head,
    head_dim,
    max_seq_len,
)

device = torch.device('cpu')
model.to(device)

LlamaModel(
  (embed_tokens): Embedding(128256, 3072)
  (decoder): LlamaDecoderLayer(
    (input_layernorm): RMSNorm((3072,), eps=1e-05, elementwise_affine=True)
    (self_attn): LlamaAttention(
      (k_proj): Linear(in_features=3072, out_features=1024, bias=False)
      (o_proj): Linear(in_features=3072, out_features=3072, bias=False)
      (q_proj): Linear(in_features=3072, out_features=3072, bias=False)
      (v_proj): Linear(in_features=3072, out_features=1024, bias=False)
    )
    (post_attention_layernorm): RMSNorm((3072,), eps=1e-05, elementwise_affine=True)
    (mlp): LlamaMlp(
      (up_proj): Linear(in_features=3072, out_features=8192, bias=False)
      (gate_proj): Linear(in_features=3072, out_features=8192, bias=False)
      (down_proj): Linear(in_features=8192, out_features=3072, bias=False)
    )
  )
  (norm): RMSNorm((3072,), eps=1e-05, elementwise_affine=True)
)

In [ ]:
from tokenizers import Tokenizer

tokenizer = Tokenizer.from_file("./my_models/tokenizer.json")


def generate(model, tokenizer, prompt, max_new_tokens):

    model.clear_kv_cache()

    encoded = tokenizer.encode(prompt)

    input_ids = torch.tensor(
        [encoded.ids],
        dtype=torch.long,
        device=device,
    )

    generated_ids = encoded.ids.copy()

    # First pass: entire prompt
    next_token = model(input_ids)

    for _ in range(max_new_tokens):

        token_id = next_token.item()
        generated_ids.append(token_id)

        if token_id == tokenizer.token_to_id("<|eot_id|>"):
            break

        # [B] -> [B, 1]
        input_ids = next_token.unsqueeze(0)

        next_token = model(input_ids)

    return tokenizer.decode(generated_ids)

In [160]:
text = generate(
    model,
    tokenizer,
    "What is AI ?",
    max_new_tokens=50,
)

print(text)

What is AI ? Artificial Intelligence is a branch of computer science that deals with the simulation of human intelligence in machines that are programmed to think like humans and mimic their actions. The term “Artificial Intelligence” was coined in 1956 by John McCarthy, an American computer
